# Grokking on modular addition

End-to-end demo: build the dataset (`data.py`), instantiate the transformer (`model.py`), train with full-batch AdamW, then plot the train/val loss and accuracy curves.

Defaults follow Nanda et al. (2023): `p=97`, addition, 30% train fraction, 1-layer transformer, `d_model=128`, AdamW with `weight_decay=1.0`, `lr=1e-3`. On a single A100/H100 the val curve typically snaps to ~100% somewhere between step 5k and 30k.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from data import GrokkingTask
from model import GrokkingTransformer, ModelConfig, num_parameters

## Config

Tweak any of these and re-run from the next cell down.

In [ ]:
# Task
P = 97
OP = "add"           # "add" | "sub" | "mul" | "div"
TRAIN_FRAC = 0.3
SEED = 0

# Optim
LR = 1e-3
WEIGHT_DECAY = 1.0   # essential for grokking; do not set to 0
BETAS = (0.9, 0.98)

# Loop
N_STEPS = 30_000
LOG_EVERY = 100

# Device: "auto" picks cuda > mps > cpu
def get_device(spec="auto"):
    if spec != "auto":
        return torch.device(spec)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device("auto")
print(f"device: {device}")

torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

## Data + model

In [ ]:
task = GrokkingTask(p=P, op=OP, train_frac=TRAIN_FRAC, seed=SEED)
train_ds, val_ds = task.split()

train_X = train_ds.tensors[0].to(device)
train_Y = train_ds.tensors[1].to(device)
val_X = val_ds.tensors[0].to(device)
val_Y = val_ds.tensors[1].to(device)

print(f"task: {OP} mod {P}  vocab={task.vocab_size}  seq_len={task.seq_len}")
print(f"train={len(train_ds)}  val={len(val_ds)}")

model_cfg = ModelConfig(vocab_size=task.vocab_size, seq_len=task.seq_len)
model = GrokkingTransformer(model_cfg).to(device)
print(f"model: {num_parameters(model):,} params")

optim = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=BETAS,
)

## Training loop

Full-batch (the dataset is tiny). Eval is also full-batch since the train+val split is only ~9k examples for `p=97`.

In [ ]:
@torch.no_grad()
def evaluate(model, X, Y):
    was_training = model.training
    model.eval()
    logits = model(X)[:, -1, :]
    loss = F.cross_entropy(logits, Y).item()
    acc = (logits.argmax(-1) == Y).float().mean().item()
    if was_training:
        model.train()
    return loss, acc

history = {
    "step": [],
    "train_loss": [], "train_acc": [],
    "val_loss": [],   "val_acc": [],
}

t0 = time.time()
for step in range(N_STEPS + 1):
    logits = model(train_X)[:, -1, :]
    loss = F.cross_entropy(logits, train_Y)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    optim.step()

    if step % LOG_EVERY == 0:
        train_loss, train_acc = evaluate(model, train_X, train_Y)
        val_loss, val_acc = evaluate(model, val_X, val_Y)
        history["step"].append(step)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if step % (LOG_EVERY * 10) == 0:
            elapsed = time.time() - t0
            print(
                f"step {step:6d}  "
                f"train loss {train_loss:.4f} acc {train_acc:.3f}  |  "
                f"val loss {val_loss:.4f} acc {val_acc:.3f}  |  "
                f"{elapsed:6.0f}s"
            )

print(f"done in {time.time() - t0:.1f}s")

## Plots

Log-scale x-axis is the conventional view: it makes the long flat "memorization" plateau and the sudden generalization snap clearly visible side by side.

In [ ]:
steps = np.array(history["step"])
# log-x can't include 0
mask = steps > 0

fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4.5))

ax_loss.plot(steps[mask], np.array(history["train_loss"])[mask], label="train")
ax_loss.plot(steps[mask], np.array(history["val_loss"])[mask],   label="val")
ax_loss.set_xscale("log")
ax_loss.set_yscale("log")
ax_loss.set_xlabel("step")
ax_loss.set_ylabel("cross-entropy loss")
ax_loss.set_title(f"Loss  ({OP} mod {P}, train_frac={TRAIN_FRAC})")
ax_loss.grid(True, which="both", alpha=0.3)
ax_loss.legend()

ax_acc.plot(steps[mask], np.array(history["train_acc"])[mask], label="train")
ax_acc.plot(steps[mask], np.array(history["val_acc"])[mask],   label="val")
ax_acc.axhline(1.0 / P, color="gray", linestyle="--", alpha=0.5, label="chance (1/p)")
ax_acc.set_xscale("log")
ax_acc.set_xlabel("step")
ax_acc.set_ylabel("accuracy")
ax_acc.set_ylim(-0.02, 1.02)
ax_acc.set_title("Accuracy")
ax_acc.grid(True, which="both", alpha=0.3)
ax_acc.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Diagnostic: at which step did val accuracy first cross 99%? (None if it didn't snap)
val_acc = np.array(history["val_acc"])
above = np.where(val_acc >= 0.99)[0]
if len(above) > 0:
    grok_step = steps[above[0]]
    print(f"val acc first hits 99% at step {grok_step}")
else:
    print(f"val acc never reached 99% (max = {val_acc.max():.3f}); try more steps")